## Задание 1

Реализуйте класс с полностью инкапсулированным состоянием, используя name mangling и property, обеспечив валидацию при изменении атрибутов и демонстрируя, как Python скрывает "приватные" данные.

In [42]:
class Encapsulated:
    def __init__(self, value):
        self.__value = None  # name mangling: stored as _Encapsulated__value
        self.value = value   # triggers the setter below

    @property
    def value(self):
        """Getter: returns the private __value attribute."""
        return self.__value

    @value.setter
    def value(self, new_value):
        """Setter: validates that new_value is a positive integer."""
        if not isinstance(new_value, int):
            raise TypeError(f'value must be int, got {type(new_value).__name__}')
        if new_value <= 0:
            raise ValueError(f'value must be positive, got {new_value}')
        self.__value = new_value

# --- demonstration ---
obj = Encapsulated(10)
print(obj.value)                  # 10  (via property getter)
print(obj._Encapsulated__value)   # 10  (direct access to name-mangled attr)

# show that the mangled name is visible in __dict__
print(obj.__dict__)               # {'_Encapsulated__value': 10}

# validation in action
try:
    obj.value = -5
except ValueError as e:
    print(e)   # value must be positive, got -5

try:
    obj.value = 'hello'
except TypeError as e:
    print(e)   # value must be int, got str

# name mangling in action: obj.__value raises AttributeError
try:
    print(obj.__value)
except AttributeError as e:
    print(e)   # 'Encapsulated' object has no attribute '__value'

10
10
{'_Encapsulated__value': 10}
value must be positive, got -5
value must be int, got str
'Encapsulated' object has no attribute '__value'


## Задание 2

Создайте иерархию классов с абстрактным базовым классом (ABC) и абстрактными методами, демонстрируя использование модуля abc. Реализуйте интерфейсы и их проверку через isinstance и issubclass.

In [43]:
from abc import ABC, abstractmethod


# ── Interface 1: Movable ─────────────────────────────────────────────────────
class Movable(ABC):
    """Interface: any movable entity must implement move()."""

    @abstractmethod
    def move(self) -> str:
        """Return how the entity moves."""
        ...


# ── Interface 2 / Abstract base: Animal ──────────────────────────────────────
class Animal(ABC):
    """Abstract base: every animal must implement sound()."""

    def __init__(self, name: str):
        self.name = name

    @abstractmethod
    def sound(self) -> str:
        """Return the sound the animal makes."""
        ...

    def describe(self) -> str:
        """Concrete method available to all subclasses."""
        move_str = self.move() if isinstance(self, Movable) else 'stays still'
        return f'{self.name} says "{self.sound()}" and {move_str}'


# ── Concrete subclasses (Animal + Movable) ────────────────────────────────────
class Dog(Animal, Movable):
    def sound(self) -> str:
        return 'woof'

    def move(self) -> str:
        return 'runs'


class Bird(Animal, Movable):
    def sound(self) -> str:
        return 'tweet'

    def move(self) -> str:
        return 'flies'


class Fish(Animal, Movable):
    def sound(self) -> str:
        return '...'

    def move(self) -> str:
        return 'swims'


# ── Partial implementation: Cat only implements sound(), not move() ───────────
class Cat(Animal, Movable):
    def sound(self) -> str:
        return 'meow'
    # move() is NOT implemented -> Cat is still abstract


# ── Cannot instantiate abstract classes ──────────────────────────────────────
print('--- abstract instantiation errors ---')
for cls, label in [(Animal, 'Animal'), (Movable, 'Movable'), (Cat, 'Cat')]:
    try:
        cls('test') if cls is not Movable else cls()
    except TypeError as e:
        print(f'{label}: {e}')


# ── Concrete instances ────────────────────────────────────────────────────────
print()
dog  = Dog('Rex')
bird = Bird('Tweety')
fish = Fish('Nemo')

for animal in (dog, bird, fish):
    print(animal.describe())


# ── isinstance checks ────────────────────────────────────────────────────────
print()
print('isinstance checks:')
print(f'dog  is Animal?  {isinstance(dog,  Animal)}')   # True
print(f'dog  is Movable? {isinstance(dog,  Movable)}')  # True
print(f'bird is Animal?  {isinstance(bird, Animal)}')   # True
print(f'dog  is Bird?    {isinstance(dog,  Bird)}')     # False


# ── issubclass checks ────────────────────────────────────────────────────────
print()
print('issubclass checks:')
print(f'Dog  subclass of Animal?  {issubclass(Dog,  Animal)}')   # True
print(f'Dog  subclass of Movable? {issubclass(Dog,  Movable)}')  # True
print(f'Bird subclass of Animal?  {issubclass(Bird, Animal)}')   # True
print(f'Dog  subclass of Bird?    {issubclass(Dog,  Bird)}')     # False
print(f'Animal subclass of ABC?   {issubclass(Animal, ABC)}')    # True
print(f'Movable subclass of ABC?  {issubclass(Movable, ABC)}')   # True

--- abstract instantiation errors ---
Animal: Can't instantiate abstract class Animal without an implementation for abstract method 'sound'
Movable: Can't instantiate abstract class Movable without an implementation for abstract method 'move'
Cat: Can't instantiate abstract class Cat without an implementation for abstract method 'move'

Rex says "woof" and runs
Tweety says "tweet" and flies
Nemo says "..." and swims

isinstance checks:
dog  is Animal?  True
dog  is Movable? True
bird is Animal?  True
dog  is Bird?    False

issubclass checks:
Dog  subclass of Animal?  True
Dog  subclass of Movable? True
Bird subclass of Animal?  True
Dog  subclass of Bird?    False
Animal subclass of ABC?   True
Movable subclass of ABC?  True


## Задание 3

Реализуйте дженерик-функцию (duck typing) с использованием протоколов (PEP 544) и типовых подсказок, чтобы показать полиморфизм без наследования.

In [44]:
from typing import Protocol, runtime_checkable, TypeVar


# ── Protocol definition (PEP 544) ─────────────────────────────────────────────
# @runtime_checkable allows isinstance() checks against the Protocol at runtime
@runtime_checkable
class Drawable(Protocol):
    """Structural interface: any object with draw() satisfies this protocol."""
    def draw(self) -> None:
        ...


# Accepts any T that satisfies Drawable and returns that same T.
T = TypeVar('T', bound=Drawable)


# ── Concrete classes — NO inheritance from Drawable ───────────────────────────
class Circle:
    """Circle implements draw() structurally — no explicit base class."""
    def draw(self) -> None:
        print('Drawing a Circle')


class Square:
    """Square implements draw() structurally — no explicit base class."""
    def draw(self) -> None:
        print('Drawing a Square')


class Triangle:
    """Triangle also satisfies Drawable without any inheritance."""
    def draw(self) -> None:
        print('Drawing a Triangle')


class NotDrawable:
    """This class has NO draw() method — does NOT satisfy Drawable."""
    def render(self) -> None:
        print('render() is not draw()')


# ── Generic paint(): TypeVar preserves the concrete type through the call ─────
def paint(obj: T) -> T:
    """Draw obj and return it — the return type is the same concrete type T."""
    obj.draw()
    return obj


# ── paint_all(): typed collection of Drawable objects ─────────────────────────
def paint_all(items: list[Drawable]) -> None:
    """Draw every item in a list[Drawable] — polymorphism without inheritance."""
    for item in items:
        item.draw()


# ── Polymorphism without inheritance ──────────────────────────────────────────
# list annotated as list[Drawable] — all three classes satisfy the protocol
shapes: list[Drawable] = [Circle(), Square(), Triangle()]
paint_all(shapes)

print()
# paint() returns the same concrete type — c is inferred as Circle by type checkers
c = paint(Circle())
print(f'paint() returned: {type(c).__name__}')  # Circle


# ── isinstance() works because of @runtime_checkable ─────────────────────────
print()
print('isinstance checks via @runtime_checkable:')
for obj in [Circle(), Square(), Triangle(), NotDrawable()]:
    print(f'{type(obj).__name__:12} is Drawable? {isinstance(obj, Drawable)}')


# ── Calling paint() with a non-Drawable raises AttributeError ─────────────────
print()
try:
    paint(NotDrawable())
except AttributeError as e:
    print(f'AttributeError: {e}')

Drawing a Circle
Drawing a Square
Drawing a Triangle

Drawing a Circle
paint() returned: Circle

isinstance checks via @runtime_checkable:
Circle       is Drawable? True
Square       is Drawable? True
Triangle     is Drawable? True
NotDrawable  is Drawable? False

AttributeError: 'NotDrawable' object has no attribute 'draw'


## Задание 4

Опишите и продемонстрируйте работу метода getattribute в отличие от getattr, реализуйте логирование всех обращений к атрибутам объекта, исследуйте возможные зацикливания.

In [ ]:
class Logger:
    def __setattr__(self, name: str, value) -> None:
        """
        Intercepts EVERY attribute write.
        Must delegate to super().__setattr__ to avoid infinite recursion.
        """
        print(f'[__setattr__]      setting "{name}" = {value!r}')
        # Correct: delegate to base implementation.
        # Wrong: self.__dict__[name] = value <- triggers __getattribute__('__dict__')
        super().__setattr__(name, value)

    def __getattribute__(self, name: str):
        """
        Intercepts EVERY attribute read.
        Must delegate to super().__getattribute__ to avoid infinite recursion.
        """
        print(f'[__getattribute__] accessing "{name}"')
        return super().__getattribute__(name)

    def __getattr__(self, name: str):
        """
        Called ONLY when __getattribute__ raises AttributeError.
        """
        print(f'[__getattr__]      "{name}" not found — raising AttributeError')
        raise AttributeError(f"'{type(self).__name__}' object has no attribute '{name}'")


obj = Logger()

# __setattr__ is called here
print('--- setting obj.x = 5 ---')
obj.x = 5
print()

# Existing attribute: __getattribute__ succeeds, __getattr__ is NOT called
print('--- reading obj.x ---')
print('obj.x =', obj.x)
print()

# Missing attribute: __getattribute__ raises AttributeError -> __getattr__ fires
# -> __getattr__ raises AttributeError -> propagates to caller
print('--- reading obj.y (missing) via direct access ---')
try:
    print(obj.y)
except AttributeError as e:
    print(f'AttributeError: {e}')
print()

# getattr(obj, name)          -> raises AttributeError if missing
# getattr(obj, name, default) -> returns default if AttributeError is raised
print('--- getattr() with default ---')
print('getattr(obj, "x")         =', getattr(obj, 'x'))          # 5
print('getattr(obj, "y", 42)     =', getattr(obj, 'y', 42))      # 42 (default)
print('getattr(obj, "z", "none") =', getattr(obj, 'z', 'none'))  # 'none' (default)
print()

# ── Demonstrate the infinite-recursion trap ───────────────────────────────────
class BadLogger:
    def __getattribute__(self, name: str):
        # BUG: self.__dict__ triggers __getattribute__('__dict__') -> recursion!
        return self.__dict__[name]   # RecursionError

bad = BadLogger()
bad.x = 42
try:
    print(bad.x)
except RecursionError:
    print('[BadLogger] RecursionError: self.__dict__ inside __getattribute__'
          ' calls __getattribute__ again!')

--- setting obj.x = 5 ---
[__setattr__]      setting "x" = 5

--- reading obj.x ---
[__getattribute__] accessing "x"
obj.x = 5

--- reading obj.y (missing) via direct access ---
[__getattribute__] accessing "y"
[__getattr__]      "y" not found — raising AttributeError
AttributeError: 'Logger' object has no attribute 'y'

--- getattr() with default ---
[__getattribute__] accessing "x"
getattr(obj, "x")         = 5
[__getattribute__] accessing "y"
[__getattr__]      "y" not found — raising AttributeError
getattr(obj, "y", 42)     = 42
[__getattribute__] accessing "z"
[__getattr__]      "z" not found — raising AttributeError
getattr(obj, "z", "none") = none

[BadLogger] RecursionError: self.__dict__ inside __getattribute__ calls __getattribute__ again!


## Задание 5

Создайте класс, который использует метакласс, объясните как метаклассы влияют на создание классов в Python, и реализуйте контролируемое изменение класса через метакласс.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# How metaclasses work
#
# In Python every class is itself an object — an instance of its metaclass.
# The default metaclass is `type`.
#
# When Python executes `class Foo(metaclass=Meta): ...` it calls:
#   Meta.__new__(mcs, name, bases, namespace)  -> allocates the class object
#   Meta.__init__(cls, name, bases, namespace) -> initialises it
#
# This lets us inspect and modify the class before it is handed to the user.
# ─────────────────────────────────────────────────────────────────────────────

class Meta(type):
    """
    Metaclass that:
    1. Logs class creation.
    2. Injects a `created_by` class attribute automatically.
    3. Enforces that all user-defined methods are lowercase.
    """

    def __new__(mcs, name: str, bases: tuple, namespace: dict):
        # ── 3. Enforce lowercase method names ────────────────────────────────
        for attr_name, attr_value in namespace.items():
            if callable(attr_value) and not attr_name.startswith('_'):
                if attr_name != attr_name.lower():
                    raise TypeError(
                        f"Method '{attr_name}' in class '{name}' must be lowercase."
                    )

        # ── 2. Inject class attribute ─────────────────────────────────────────
        namespace['created_by'] = 'Meta'

        # ── 1. Log and delegate to type.__new__ ───────────────────────────────
        print(f'[Meta.__new__]  creating class "{name}"')
        print(f'  bases     : {bases}')
        print(f'  attributes: {[k for k in namespace if not k.startswith("__")]}')
        cls = super().__new__(mcs, name, bases, namespace)
        return cls

    def __init__(cls, name: str, bases: tuple, namespace: dict):
        print(f'[Meta.__init__] initialising class "{name}"')
        super().__init__(name, bases, namespace)


# ── GoodClass: follows all rules ─────────────────────────────────────────────
print('=== defining GoodClass ===')
class GoodClass(metaclass=Meta):
    def greet(self):
        return 'hello from GoodClass'

    def compute(self, x: int) -> int:
        return x * 2

print()
obj = GoodClass()
print('obj.greet()      =', obj.greet())
print('obj.compute(21)  =', obj.compute(21))
print('GoodClass.created_by =', GoodClass.created_by)
print('type(GoodClass)      =', type(GoodClass))   # <class '__main__.Meta'>


# ── BadClass: violates the lowercase rule ─────────────────────────────────────
print()
print('=== defining BadClass (uppercase method) ===')
try:
    class BadClass(metaclass=Meta):
        def BadMethod(self):   # uppercase -> Meta rejects it
            pass
except TypeError as e:
    print(f'TypeError: {e}')

# ── Show that GoodClass is an instance of Meta ────────────────────────────────
print(isinstance(GoodClass, Meta))   # True
print(issubclass(GoodClass, object)) # True
print('type(GoodClass) =', type(GoodClass))
print('type(obj) =', type(obj))

=== defining GoodClass ===
[Meta.__new__]  creating class "GoodClass"
  bases     : ()
  attributes: ['greet', 'compute', 'created_by']
[Meta.__init__] initialising class "GoodClass"

obj.greet()      = hello from GoodClass
obj.compute(21)  = 42
GoodClass.created_by = Meta
type(GoodClass)      = <class '__main__.Meta'>

=== defining BadClass (uppercase method) ===
TypeError: Method 'BadMethod' in class 'BadClass' must be lowercase.
True
True
type(GoodClass) = <class '__main__.Meta'>
type(obj) = <class '__main__.GoodClass'>


## Задание 6

Реализуйте класс с дескрипторами данных и неданных, объясните разницу между ними и механизм вызова методов get, set, delete, особенно при наследовании.

In [47]:
# ─────────────────────────────────────────────────────────────────────────────
# A descriptor is any object that defines at least one of:
#   __get__(self, obj, objtype=None)  -> called on attribute READ
#   __set__(self, obj, value)         -> called on attribute WRITE
#   __delete__(self, obj)             -> called on attribute DELETE
#
# Data descriptor: has __set__ or __delete__ (usually also __get__)
#   Priority: data descriptor > instance __dict__ > non-data descriptor
#
# Non-data descriptor: has __get__ only
#   Priority: instance __dict__ > non-data descriptor
#
# ── READ  obj.attr  (object.__getattribute__) ────────────────────────────────
#   1. Walk MRO; if a DATA descriptor is found -> descriptor.__get__(obj, type)
#   2. Check obj.__dict__; if key present -> return value
#   3. Walk MRO; if a NON-DATA descriptor is found -> descriptor.__get__(obj, type)
#      or return the plain class attribute
#   4. Nothing found -> AttributeError is raised by __getattribute__;
#      if __getattr__ is defined on the class, Python calls it as a last resort
#
# ── WRITE  obj.attr = value  (object.__setattr__) ────────────────────────────
#   1. Walk MRO; if a descriptor with __set__ is found
#      -> descriptor.__set__(obj, value)  [data descriptor intercepts the write]
#   2. Otherwise -> value is stored directly in obj.__dict__[attr]
#      (non-data descriptor is bypassed; instance dict entry shadows it)
#
# ── DELETE  del obj.attr  (object.__delattr__) ───────────────────────────────
#   1. Walk MRO; if a descriptor with __delete__ is found
#      -> descriptor.__delete__(obj)  [data descriptor intercepts the delete]
#   2. Otherwise -> del obj.__dict__[attr]
#      if the key is absent -> AttributeError
# ─────────────────────────────────────────────────────────────────────────────


# ── DATA DESCRIPTOR ──────────────────────────────────────────────────────────
# Defines __get__, __set__, and __delete__.
# Stores the value in the *instance* dict under a private key so that
# multiple instances each have their own storage.
class DataDescriptor:
    """
    Data descriptor: defines __get__, __set__, __delete__.
    Because __set__ is present it takes priority over the instance __dict__.
    Validates that the stored value is a positive integer.
    """

    def __set_name__(self, owner, name: str) -> None:
        # Called automatically when the descriptor is assigned as a class attr.
        # We use a mangled private key so different descriptors don't collide.
        self._storage_name = f'_dd_{name}'

    def __get__(self, obj, objtype=None):
        """
        obj      — the instance (None when accessed on the class itself)
        objtype  — the owner class (always provided)
        """
        if obj is None:
            # Accessed via the class, e.g. MyClass.data_desc -> return descriptor itself
            return self
        value = obj.__dict__.get(self._storage_name, None)
        print(f'[DataDescriptor.__get__]    reading "{self._storage_name}" -> {value!r}')
        return value

    def __set__(self, obj, value) -> None:
        """
        Because __set__ is defined, this is a DATA descriptor.
        It intercepts obj.attr = value and stores it under the private key.
        """
        if not isinstance(value, int) or value <= 0:
            raise ValueError(f'DataDescriptor requires a positive int, got {value!r}')
        print(f'[DataDescriptor.__set__]    writing "{self._storage_name}" = {value!r}')
        obj.__dict__[self._storage_name] = value

    def __delete__(self, obj) -> None:
        print(f'[DataDescriptor.__delete__] deleting "{self._storage_name}"')
        obj.__dict__.pop(self._storage_name, None)


# ── NON-DATA DESCRIPTOR ───────────────────────────────────────────────────────
# Defines ONLY __get__ — no __set__, no __delete__.
# The instance __dict__ takes priority: if obj.__dict__['attr'] exists,
# the descriptor is bypassed entirely.
class NonDataDescriptor:
    """
    Non-data descriptor: defines only __get__.
    Instance __dict__ shadows it — assigning obj.non_data_desc = x
    stores x directly in obj.__dict__ and the descriptor is never called again.
    """

    def __get__(self, obj, objtype=None):
        if obj is None:
            return self
        print(f'[NonDataDescriptor.__get__] called on {type(obj).__name__} instance')
        return 'non-data default value'


# ── Owner class ───────────────────────────────────────────────────────────────
class MyClass2:
    data_desc     = DataDescriptor()     # data descriptor
    non_data_desc = NonDataDescriptor()  # non-data descriptor


# ── Basic usage ───────────────────────────────────────────────────────────────
print('=' * 60)
print('1. Basic descriptor usage')
print('=' * 60)

obj = MyClass2()

# __set__ is called -> value stored in obj.__dict__['_dd_data_desc']
obj.data_desc = 10
# __get__ is called -> reads from obj.__dict__['_dd_data_desc']
print('obj.data_desc     =', obj.data_desc)

# NonDataDescriptor.__get__ is called (nothing in instance __dict__ yet)
print('obj.non_data_desc =', obj.non_data_desc)


# ── DATA descriptor vs instance __dict__ ─────────────────────────────────────
print()
print('=' * 60)
print('2. Data descriptor WINS over instance __dict__')
print('=' * 60)

# Bypass __set__ and write directly into instance __dict__
obj.__dict__['data_desc'] = 999
print('obj.__dict__["data_desc"] set to 999 directly')
# __get__ is still called because DataDescriptor is a DATA descriptor
# -> it reads from '_dd_data_desc', NOT from 'data_desc'
print('obj.data_desc =', obj.data_desc)   # still 10, not 999
print('obj.__dict__  =', obj.__dict__)     # both keys visible


# ── NON-DATA descriptor vs instance __dict__ ─────────────────────────────────
print()
print('=' * 60)
print('3. Instance __dict__ WINS over non-data descriptor')
print('=' * 60)

obj2 = MyClass2()
print('Before assignment — descriptor is called:')
print('obj2.non_data_desc =', obj2.non_data_desc)  # descriptor __get__

# Normal assignment goes straight to instance __dict__ (no __set__ to intercept)
obj2.non_data_desc = 'shadowed by instance dict'
print('After  assignment — descriptor is BYPASSED:')
print('obj2.non_data_desc =', obj2.non_data_desc)  # reads from __dict__
print('obj2.__dict__      =', obj2.__dict__)


# ── __delete__ on data descriptor ────────────────────────────────────────────
print()
print('=' * 60)
print('4. __delete__ on data descriptor')
print('=' * 60)

obj3 = MyClass2()
obj3.data_desc = 42
print('Before delete:', obj3.data_desc)
del obj3.data_desc                        # calls DataDescriptor.__delete__
print('After  delete:', obj3.data_desc)   # None (key removed from __dict__)


# ── Validation in data descriptor ────────────────────────────────────────────
print()
print('=' * 60)
print('5. Validation enforced by data descriptor __set__')
print('=' * 60)

obj4 = MyClass2()
try:
    obj4.data_desc = -5
except ValueError as e:
    print(f'ValueError: {e}')
try:
    obj4.data_desc = 'hello'
except ValueError as e:
    print(f'ValueError: {e}')


# ── Inheritance: descriptor is inherited through MRO ─────────────────────────
print()
print('=' * 60)
print('6. Descriptor inheritance through MRO')
print('=' * 60)

class ChildClass(MyClass2):
    """Inherits both descriptors from MyClass2."""
    pass

class GrandChildClass(ChildClass):
    """Overrides data_desc with a plain class attribute (shadows the descriptor)."""
    data_desc = 'plain class attr — descriptor is gone at this level'

child = ChildClass()
child.data_desc = 7          # DataDescriptor.__set__ is called (inherited)
print('child.data_desc     =', child.data_desc)      # DataDescriptor.__get__
print('child.non_data_desc =', child.non_data_desc)  # NonDataDescriptor.__get__

gc = GrandChildClass()
# GrandChildClass.data_desc is a plain str, not a descriptor -> no __get__/__set__
print('GrandChildClass.data_desc =', GrandChildClass.data_desc)
gc.data_desc = 'instance value'   # goes straight to gc.__dict__
print('gc.data_desc (instance)   =', gc.data_desc)

print()
print('MRO of GrandChildClass:', [c.__name__ for c in GrandChildClass.__mro__])


# ── Accessing descriptor on the class (obj is None) ──────────────────────────
print()
print('=' * 60)
print('7. Accessing descriptor via the class (obj=None)')
print('=' * 60)

print('MyClass2.data_desc     =', MyClass2.data_desc)      # returns descriptor itself
print('MyClass2.non_data_desc =', MyClass2.non_data_desc)  # returns descriptor itself


# ── __getattr__: last resort when all other lookup steps fail ─────────────────
# object.__getattribute__ raises AttributeError when:
#   - no data descriptor found in MRO
#   - name not in obj.__dict__
#   - no non-data descriptor / plain class attr found in MRO
# Python then calls type(obj).__getattr__(obj, name) if it is defined.
# __getattr__ is not called when the attribute is found by normal lookup.
print()
print('=' * 60)
print('8. __getattr__ as the last resort in the lookup chain')
print('=' * 60)

class MyClass3(MyClass2):
    """
    Extends MyClass2 with __getattr__ so that any unknown attribute
    returns a computed fallback instead of raising AttributeError.
    __getattr__ is called ONLY after all other lookup steps have failed:
      data descriptor -> instance __dict__ -> non-data descriptor -> __getattr__
    """

    def __getattr__(self, name: str):
        # This is reached only when normal lookup found nothing.
        print(f'[__getattr__] "{name}" not found by normal lookup — returning fallback')
        return f'<fallback for {name!r}>'

obj5 = MyClass3()

# Step 1: data descriptor wins — __getattr__ is NOT called
obj5.data_desc = 99
print('obj5.data_desc      =', obj5.data_desc)       # DataDescriptor.__get__

# Step 2: instance __dict__ wins — __getattr__ is NOT called
obj5.__dict__['cached'] = 'from instance dict'
print('obj5.cached         =', obj5.cached)           # reads __dict__ directly

# Step 3: non-data descriptor wins — __getattr__ is NOT called
print('obj5.non_data_desc  =', obj5.non_data_desc)    # NonDataDescriptor.__get__

# Step 4: nothing found -> __getattr__ IS called as last resort
print('obj5.unknown_attr   =', obj5.unknown_attr)     # __getattr__ fires
print('obj5.another_miss   =', obj5.another_miss)     # __getattr__ fires again

# Confirm: without __getattr__ the same access raises AttributeError
plain = MyClass2()
try:
    _ = plain.unknown_attr
except AttributeError as e:
    print(f'AttributeError on MyClass2 (no __getattr__): {e}')

1. Basic descriptor usage
[DataDescriptor.__set__]    writing "_dd_data_desc" = 10
[DataDescriptor.__get__]    reading "_dd_data_desc" -> 10
obj.data_desc     = 10
[NonDataDescriptor.__get__] called on MyClass2 instance
obj.non_data_desc = non-data default value

2. Data descriptor WINS over instance __dict__
obj.__dict__["data_desc"] set to 999 directly
[DataDescriptor.__get__]    reading "_dd_data_desc" -> 10
obj.data_desc = 10
obj.__dict__  = {'_dd_data_desc': 10, 'data_desc': 999}

3. Instance __dict__ WINS over non-data descriptor
Before assignment — descriptor is called:
[NonDataDescriptor.__get__] called on MyClass2 instance
obj2.non_data_desc = non-data default value
After  assignment — descriptor is BYPASSED:
obj2.non_data_desc = shadowed by instance dict
obj2.__dict__      = {'non_data_desc': 'shadowed by instance dict'}

4. __delete__ on data descriptor
[DataDescriptor.__set__]    writing "_dd_data_desc" = 42
[DataDescriptor.__get__]    reading "_dd_data_desc" -> 42
Before d

## Задание 7

Напишите класс с поддержкой множественного наследования, демонстрирующий работу C3-линеаризации MRO на сложном примере с 3+ уровнями наследования и пересечениями.

In [48]:

# ── Level 1 ───────────────────────────────────────────────────────────────────
class A:
    def method(self):
        print(f'  A.method  (self type: {type(self).__name__})')
        # super() here would call object.method — which doesn't exist,
        # so we stop the cooperative chain here.

    def whoami(self) -> str:
        return 'A'


# ── Level 2 ───────────────────────────────────────────────────────────────────
class B(A):
    def method(self):
        print(f'  B.method  (self type: {type(self).__name__})')
        super().method()   # cooperative: follows MRO, not just A

    def whoami(self) -> str:
        return f'B -> {super().whoami()}'


class C(A):
    def method(self):
        print(f'  C.method  (self type: {type(self).__name__})')
        super().method()

    def whoami(self) -> str:
        return f'C -> {super().whoami()}'


# ── Level 3 ───────────────────────────────────────────────────────────────────
class D(B, C):
    """Classic diamond: inherits from both B and C."""
    def method(self):
        print(f'  D.method  (self type: {type(self).__name__})')
        super().method()

    def whoami(self) -> str:
        return f'D -> {super().whoami()}'


class E(B):
    def method(self):
        print(f'  E.method  (self type: {type(self).__name__})')
        super().method()

    def whoami(self) -> str:
        return f'E -> {super().whoami()}'


class F(C):
    def method(self):
        print(f'  F.method  (self type: {type(self).__name__})')
        super().method()

    def whoami(self) -> str:
        return f'F -> {super().whoami()}'


# ── Level 4 ───────────────────────────────────────────────────────────────────
class G(D, E, F):
    """
    G sits at the bottom of a 4-level diamond-plus hierarchy.
    Its MRO is computed by C3 as: G -> D -> E -> B -> F -> C -> A -> object
    """
    def method(self):
        print(f'  G.method  (self type: {type(self).__name__})')
        super().method()   # kicks off the cooperative chain

    def whoami(self) -> str:
        return f'G -> {super().whoami()}'


# ── MRO inspection ────────────────────────────────────────────────────────────
print('=' * 60)
print('MRO for each class (C3 linearization)')
print('=' * 60)
for cls in (A, B, C, D, E, F, G):
    mro_names = [c.__name__ for c in cls.__mro__]
    print(f'{cls.__name__:>3}.__mro__ = {mro_names}')


# ── Cooperative super() chain ─────────────────────────────────────────────────
print()
print('=' * 60)
print('Cooperative super() chain: g.method()')
print('=' * 60)
print('Call order follows MRO — each class calls super().method():')
g = G()
g.method()
# Expected output order: G -> D -> E -> B -> F -> C -> A


# ── whoami() traces the full MRO path ─────────────────────────────────────────
print()
print('=' * 60)
print('whoami() traces the cooperative super() path')
print('=' * 60)
print('g.whoami() =', g.whoami())
# Expected: G -> D -> E -> B -> F -> C -> A


# ── isinstance / issubclass across the hierarchy ──────────────────────────────
print()
print('=' * 60)
print('isinstance / issubclass checks')
print('=' * 60)
for base in (G, D, E, F, B, C, A, object):
    print(f'isinstance(g, {base.__name__}) = {isinstance(g, base)}')

print()
# G is a subclass of every class in its MRO
for base in (D, E, F, B, C, A):
    print(f'issubclass(G, {base.__name__}) = {issubclass(G, base)}')

print (G.__mro__)


# ── Demonstrate that A.method is called exactly ONCE ─────────────────────────
# Without cooperative super() (i.e. if each class called A.method directly)
# A.method would be invoked multiple times.  With super() + C3 MRO it fires
# exactly once, at the end of the chain.
print()
print('=' * 60)
print('Verify A.method is called exactly once (no duplicate calls)')
print('=' * 60)
call_log: list[str] = []

class A2:
    def method(self): call_log.append('A2'); super().method() if hasattr(super(), 'method') else None

class B2(A2):
    def method(self): call_log.append('B2'); super().method()

class C2(A2):
    def method(self): call_log.append('C2'); super().method()

class D2(B2, C2):
    def method(self): call_log.append('D2'); super().method()

D2().method()
print('Call order:', call_log)   # ['D2', 'B2', 'C2', 'A2'] — A2 appears once
assert call_log.count('A2') == 1, 'A2 must be called exactly once!'
print('A2 called exactly once: OK')

MRO for each class (C3 linearization)
  A.__mro__ = ['A', 'object']
  B.__mro__ = ['B', 'A', 'object']
  C.__mro__ = ['C', 'A', 'object']
  D.__mro__ = ['D', 'B', 'C', 'A', 'object']
  E.__mro__ = ['E', 'B', 'A', 'object']
  F.__mro__ = ['F', 'C', 'A', 'object']
  G.__mro__ = ['G', 'D', 'E', 'B', 'F', 'C', 'A', 'object']

Cooperative super() chain: g.method()
Call order follows MRO — each class calls super().method():
  G.method  (self type: G)
  D.method  (self type: G)
  E.method  (self type: G)
  B.method  (self type: G)
  F.method  (self type: G)
  C.method  (self type: G)
  A.method  (self type: G)

whoami() traces the cooperative super() path
g.whoami() = G -> D -> E -> B -> F -> C -> A

isinstance / issubclass checks
isinstance(g, G) = True
isinstance(g, D) = True
isinstance(g, E) = True
isinstance(g, F) = True
isinstance(g, B) = True
isinstance(g, C) = True
isinstance(g, A) = True
isinstance(g, object) = True

issubclass(G, D) = True
issubclass(G, E) = True
issubclass(G, F) = T

## Задание 8

Реализуйте класс с пользовательскими dunder-методами: new, init, call, del, repr, str, и объясните, как Python вызывает их в цикле жизни объекта.

In [49]:
# ── Object life-cycle ────────────────────────────────────────────────────────
#
# 1. ClassName(...)  ->  type.__call__(ClassName, ...)
#      a) ClassName.__new__(cls, ...)   allocates & returns the raw instance
#      b) ClassName.__init__(self, ...) initialises the instance in-place
#
# 2. print(obj)      ->  str(obj)  ->  obj.__str__()
#    repr(obj)       ->  obj.__repr__()
#    (if __str__ is absent, Python falls back to __repr__)
#
# 3. obj(...)        ->  obj.__call__(...)
#    Makes the instance callable, just like a function.
#
# 4. del obj         ->  decrements reference count
#    When the count reaches 0, CPython calls obj.__del__() before freeing memory.
#
# ─────────────────────────────────────────────────────────────────────────────


class LifeCycle:
    """
    Demonstrates every stage of a Python object's life cycle:

    __new__   — allocation   (called before __init__)
    __init__  — initialisation
    __repr__  — unambiguous developer representation
    __str__   — human-readable string
    __call__  — makes the instance callable
    __del__   — finalisation (called when reference count drops to 0)
    """

    # ── 1. __new__: allocate the raw instance ─────────────────────────────────
    # Called by type.__call__ BEFORE __init__.
    # Must return an instance of cls (or a subclass); if it doesn't return
    # an instance of cls, __init__ is skipped entirely.
    def __new__(cls, value: int = 0):
        print(f'[__new__]  allocating a new {cls.__name__} instance (cls={cls.__name__})')
        instance = super().__new__(cls)   # delegate to object.__new__
        return instance

    # ── 2. __init__: initialise the instance ──────────────────────────────────
    # Called by type.__call__ immediately after __new__ returns an instance.
    # Receives the same arguments as __new__ (except cls -> self).
    def __init__(self, value: int = 0):
        print(f'[__init__] initialising instance; value={value}')
        self.value = value
        self._call_count = 0   # tracks how many times the instance is called

    # ── 3. __repr__: unambiguous developer representation ─────────────────────
    # Called by repr(obj), and as a fallback when __str__ is absent.
    # Should ideally return a string that could recreate the object.
    def __repr__(self) -> str:
        return f'LifeCycle(value={self.value!r}, calls={self._call_count})'

    # ── 4. __str__: human-readable representation ─────────────────────────────
    # Called by str(obj) and print(obj).
    # Falls back to __repr__ if not defined.
    def __str__(self) -> str:
        return (
            f'LifeCycle instance | value={self.value} | '
            f'called {self._call_count} time(s)'
        )

    # ── 5. __call__: make the instance callable ────────────────────────────────
    # Invoked when the instance is used as obj(...).  After this is defined,
    # callable(obj) returns True.
    def __call__(self, *args, **kwargs) -> None:
        self._call_count += 1
        ordinal = {1: '1st', 2: '2nd', 3: '3rd'}.get(self._call_count,
                                                       f'{self._call_count}th')
        print(
            f'[__call__] instance called for the {ordinal} time '
            f'with args={args} kwargs={kwargs}'
        )

    # ── 6. __del__: finaliser ─────────────────────────────────────────────────
    # Called by CPython when the reference count of the object drops to zero.
    def __del__(self) -> None:
        value = getattr(self, 'value', '<not set>')
        calls = getattr(self, '_call_count', '<not set>')
        print(
            f'[__del__]  finalising instance; '
            f'value={value}, was called {calls} time(s)'
        )


# ── Demonstration ─────────────────────────────────────────────────────────────
print()
# type.__call__ -> __new__ then __init__
obj = LifeCycle(42)

print()
print('--- repr(obj) ---')
print(repr(obj))   # __repr__

print()
print('--- str(obj) / print(obj) ---')
print(obj)         # __str__

print()
print('--- obj() ---')
obj()              # __call__ (1st time)

print()
print("--- obj('hello', key='world') ---")
obj('hello', key='world')   # __call__ (2nd time)

print()
print('--- str(obj) after two calls ---')
print(obj)         # __str__ again — call count updated

print()
print('--- del obj ---')
del obj            # reference count -> 0 -> __del__

print()
print('--- extra reference demo ---')
obj = LifeCycle(100)
alias = obj
obj('hello', key='world')
del obj              # __del__ not called yet
print(alias)
del alias            # now object can be finalized


[__new__]  allocating a new LifeCycle instance (cls=LifeCycle)
[__init__] initialising instance; value=42

--- repr(obj) ---
LifeCycle(value=42, calls=0)

--- str(obj) / print(obj) ---
LifeCycle instance | value=42 | called 0 time(s)

--- obj() ---
[__call__] instance called for the 1st time with args=() kwargs={}

--- obj('hello', key='world') ---
[__call__] instance called for the 2nd time with args=('hello',) kwargs={'key': 'world'}

--- str(obj) after two calls ---
LifeCycle instance | value=42 | called 2 time(s)

--- del obj ---
[__del__]  finalising instance; value=42, was called 2 time(s)

--- extra reference demo ---
[__new__]  allocating a new LifeCycle instance (cls=LifeCycle)
[__init__] initialising instance; value=100
[__call__] instance called for the 1st time with args=('hello',) kwargs={'key': 'world'}
LifeCycle instance | value=100 | called 1 time(s)
[__del__]  finalising instance; value=100, was called 1 time(s)


## Задание 9

Создайте контекстный менеджер с помощью специальных методов enter и exit, используйте его вместе с классом, в котором присутствуют методы с разграничением прав доступа.

https://habr.com/ru/articles/739326/

In [ ]:
class Access:
    """
    Combines a context manager with access-level enforcement.

    Outside the 'with' block:
      - public_info()  is always available
      - secret         property returns a masked value '***'
      - read_secret()  raises PermissionError
      - write_secret() raises PermissionError

    Inside the 'with' block (privileged mode):
      - __enter__ activates privileged mode
      - secret         property returns the real value
      - read_secret()  returns the real value
      - write_secret() updates the stored secret
      - __exit__  deactivates privileged mode (even if an exception occurred)
    """

    # ── Private storage (name-mangled: _Access__secret, _Access__privileged) ──
    def __init__(self, secret: str = 'top-secret-data', name: str = 'vault'):
        self.__secret: str = secret          # private — never directly accessible
        self.__privileged: bool = False      # private flag: are we inside 'with'?
        self._name: str = name               # protected by convention
        self._version: str = '1.0'           # protected by convention

    # ── Helper: enforce privileged mode ───────────────────────────────────────
    def _require_privileged(self, method_name: str) -> None:
        """Raise PermissionError if not in privileged (context-manager) mode."""
        if not self.__privileged:
            raise PermissionError(
                f"Access denied: {method_name!r} requires privileged mode"
            )

    # ── Public method: always accessible ──────────────────────────────────────
    def public_info(self) -> str:
        """Public: returns non-sensitive metadata."""
        return f'Public info: name={self._name}, version={self._version}'

    # ── Protected property: masked outside 'with', real value inside ──────────
    @property
    def secret(self) -> str:
        """
        Outside privileged mode -> returns '***' (masked).
        Inside  privileged mode -> returns the real secret.
        """
        if self.__privileged:
            return self.__secret
        return '***'

    # ── Protected methods: only callable inside 'with' block ──────────────────
    def read_secret(self) -> str:
        """Protected: read the real secret — requires privileged mode."""
        self._require_privileged('read_secret')
        return self.__secret

    def write_secret(self, new_secret: str) -> None:
        """Protected: overwrite the secret — requires privileged mode."""
        self._require_privileged('write_secret')
        if not isinstance(new_secret, str) or not new_secret:
            raise ValueError('secret must be a non-empty string')
        self.__secret = new_secret

    # ── Context manager protocol ───────────────────────────────────────────────
    def __enter__(self) -> str:
        """
        Activate privileged mode.
        Returns the real secret so it can be bound with 'as secret_value'.
        """
        self.__privileged = True
        print('[__enter__] Privileged mode ACTIVATED')
        return self 

    def __exit__(self, exc_type, exc_val, tb) -> bool:
        """
        Deactivate privileged mode — always, even if an exception occurred.

        exc_type / exc_val / tb — exception info (all None if no exception).
        Returning False (or None) re-raises the exception.
        Returning True suppresses it.
        """
        self.__privileged = False
        if exc_type is None:
            print('[__exit__]  Privileged mode DEACTIVATED')
        else:
            print(f'[__exit__]  Privileged mode DEACTIVATED '
                  f'(exception: {exc_val!r})')
        return False   # do NOT suppress exceptions by default


# ── Demonstration ─────────────────────────────────────────────────────────────
print('=' * 60)
print('Access control class with context manager')
print('=' * 60)

obj = Access()

# ── Outside the context manager ───────────────────────────────────────────────
print()
print("--- Outside context manager ---")
print('obj.secret (masked) =', obj.secret)          # '***'
print('obj.public_info()   =', obj.public_info())   # always works

try:
    obj.read_secret()                               # PermissionError
except PermissionError as e:
    print(f'PermissionError: {e}')

try:
    obj.write_secret('hacked')                     # PermissionError
except PermissionError as e:
    print(f'PermissionError: {e}')

# ── Inside the context manager ────────────────────────────────────────────────
print()
print("--- Inside 'with obj as secret_value' block ---")
with obj:
    print('obj.secret (unmasked)         =', obj.secret)     # real secret via property
    print('obj.read_secret()             =', obj.read_secret())
    obj.write_secret('new-secret')
    print("obj.write_secret('new-secret') -> OK")
    print('obj.read_secret() after write =', obj.read_secret())

# ── After the context manager ─────────────────────────────────────────────────
print()
print('--- After context manager ---')
print('obj.secret (masked again) =', obj.secret)   # '***' again
try:
    obj.read_secret()
except PermissionError as e:
    print(f'PermissionError: {e}')

# ── Exception inside 'with' block — __exit__ still fires ─────────────────────
print()
print("--- Exception inside 'with' block is NOT suppressed ---")
try:
    with obj as sv:
        raise ValueError('boom')   # __exit__ is called with exc info
except ValueError as e:
    print(f'Exception was NOT suppressed: {e}')

# ── Suppressing exception: subclass that returns True from __exit__ ───────────
print()
print('--- Suppressing exception demo ---')

class SuppressingAccess(Access):
    """Variant that silently swallows RuntimeError inside the 'with' block."""
    def __exit__(self, exc_type, exc_val, tb) -> bool:
        suppress = super().__exit__(exc_type, exc_val, tb)
        if exc_type is not None and issubclass(exc_type, RuntimeError):
            return True   # suppress RuntimeError
        return suppress

sa = SuppressingAccess(secret='suppressed-secret')
with sa:
    raise RuntimeError('suppressed!')   # will be swallowed
print('Exception was suppressed — execution continues here')

Access control class with context manager

--- Outside context manager ---
obj.secret (masked) = ***
obj.public_info()   = Public info: name=vault, version=1.0
PermissionError: Access denied: 'read_secret' requires privileged mode
PermissionError: Access denied: 'write_secret' requires privileged mode

--- Inside 'with obj as secret_value' block ---
[__enter__] Privileged mode ACTIVATED
obj.secret (unmasked)         = top-secret-data
obj.read_secret()             = top-secret-data
obj.write_secret('new-secret') -> OK
obj.read_secret() after write = new-secret
[__exit__]  Privileged mode DEACTIVATED

--- After context manager ---
obj.secret (masked again) = ***
PermissionError: Access denied: 'read_secret' requires privileged mode

--- Exception inside 'with' block is NOT suppressed ---
[__enter__] Privileged mode ACTIVATED
[__exit__]  Privileged mode DEACTIVATED (exception: ValueError('boom'))
Exception was NOT suppressed: boom

--- Suppressing exception demo ---
[__enter__] Privileged

## Задание 10

Создайте класс, объекты которого могут быть отслежены с помощью слабых ссылок (weakref). Реализуйте систему, которая хранит слабые ссылки на все созданные объекты класса и автоматически удаляет их из списка при уничтожении объектов. Продемонстрируйте это поведение, выводя текущее количество живых объектов.

In [ ]:
import weakref
import gc


class Tracked:
    """
    Every instance registers itself in the class-level WeakSet on creation.
    When an instance is garbage-collected the WeakSet removes it automatically
    — no manual bookkeeping required.

    weakref.WeakSet is chosen over a plain list of weakref.ref() objects
    because it handles cleanup transparently and supports len().
    """

    # Class-level registry: holds *weak* references to all live instances.
    # When an instance is collected, WeakSet silently drops the dead entry.
    _registry: weakref.WeakSet = weakref.WeakSet()

    def __init__(self, name: str) -> None:
        self.name = name
        # Register this instance; WeakSet stores only a weak reference,
        # so this line does NOT extend the object's lifetime.
        Tracked._registry.add(self)
        print(f'  [+] Created  {self!r:30s}  | live: {Tracked.live_count()}')

    def __repr__(self) -> str:
        return f'Tracked({self.name!r})'

    def __del__(self) -> None:
        # Called by CPython when the reference count drops to zero.
        # At this point the object is still valid inside __del__.
        # WeakSet will remove the entry on its own, but we print here
        # to show the exact moment of destruction.
        print(f'  [-] Destroyed {self!r}')

    # ── Class-level helpers ───────────────────────────────────────────────────
    @classmethod
    def live_count(cls) -> int:
        """Return the number of currently live instances."""
        return len(cls._registry)

    @classmethod
    def live_instances(cls) -> list['Tracked']:
        """Return a snapshot list of all currently live instances."""
        # list() materialises the WeakSet into strong references for the
        # duration of this call, preventing collection mid-iteration.
        return list(cls._registry)


# ── Helper ────────────────────────────────────────────────────────────────────
def show_live(label: str = '') -> None:
    instances = Tracked.live_instances()
    names = ', '.join(repr(o) for o in instances) or '<none>'
    print(f'  >>> live={Tracked.live_count()}  [{names}]  {label}')


# ─────────────────────────────────────────────────────────────────────────────
# Demo 1: basic creation and deletion
# ─────────────────────────────────────────────────────────────────────────────
print('=' * 60)
print('Demo 1: basic creation and deletion')
print('=' * 60)

a = Tracked('alpha')
b = Tracked('beta')
c = Tracked('gamma')
show_live('after creating a, b, c')

print()
del b                   # drops the only strong reference -> object is collected
show_live('after del b')

print()
del a
show_live('after del a')

print()
del c
show_live('after del c')


# ─────────────────────────────────────────────────────────────────────────────
# Demo 2: alias — object survives as long as ANY strong reference exists
# ─────────────────────────────────────────────────────────────────────────────
print()
print('=' * 60)
print('Demo 2: alias — object survives while any strong ref exists')
print('=' * 60)

x = Tracked('x-ray')
alias = x               # second strong reference
show_live('after creating x (alias also holds it)')

print()
del x                   # one strong ref gone, but alias still holds it
show_live('after del x  (alias still alive)')

print()
del alias               # last strong ref gone -> collected now
show_live('after del alias')


# ─────────────────────────────────────────────────────────────────────────────
# Demo 3: low-level weakref.ref with a finalisation callback
# ─────────────────────────────────────────────────────────────────────────────
print()
print('=' * 60)
print('Demo 3: low-level weakref.ref with callback')
print('=' * 60)

def on_collected(dead_ref: weakref.ref) -> None:
    """Callback invoked by the GC when the referent is about to be freed."""
    print(f'  [callback] weak ref {dead_ref} is now dead')

obj = Tracked('omega')
weak = weakref.ref(obj, on_collected)   # low-level weak reference

print(f'  weak() before deletion : {weak()}')   # returns the live object
show_live('before del obj')

print()
del obj
print(f'  weak() after  deletion : {weak()}')   # returns None
show_live('after del obj')


# ─────────────────────────────────────────────────────────────────────────────
# Demo 4: objects inside a local scope — collected when scope exits
# ─────────────────────────────────────────────────────────────────────────────
print()
print('=' * 60)
print('Demo 4: objects created inside a function scope')
print('=' * 60)

def create_batch(n: int) -> None:
    """Create n Tracked objects; they are local and die when the function returns."""
    batch = [Tracked(f'item-{i}') for i in range(n)]
    show_live(f'inside create_batch({n})')
    # batch goes out of scope here -> all objects collected

show_live('before create_batch')
create_batch(4)
gc.collect()            # ensure CPython collected the locals (needed in some envs)
show_live('after  create_batch — all locals collected')

Demo 1: basic creation and deletion
  [+] Created  Tracked('alpha')                | live: 1
  [+] Created  Tracked('beta')                 | live: 2
  [+] Created  Tracked('gamma')                | live: 3
  >>> live=3  [Tracked('gamma'), Tracked('beta'), Tracked('alpha')]  after creating a, b, c

  [-] Destroyed Tracked('beta')
  >>> live=2  [Tracked('gamma'), Tracked('alpha')]  after del b

  [-] Destroyed Tracked('alpha')
  >>> live=1  [Tracked('gamma')]  after del a

  [-] Destroyed Tracked('gamma')
  >>> live=0  [<none>]  after del c

Demo 2: alias — object survives while any strong ref exists
  [+] Created  Tracked('x-ray')                | live: 1
  >>> live=1  [Tracked('x-ray')]  after creating x (alias also holds it)

  >>> live=1  [Tracked('x-ray')]  after del x  (alias still alive)

  [-] Destroyed Tracked('x-ray')
  >>> live=0  [<none>]  after del alias

Demo 3: low-level weakref.ref with callback
  [+] Created  Tracked('omega')                | live: 1
  weak() before del